# Basic plotting with new weather observations
- testing to see if the graphs look correct
- to see if they will actually plot
- seeing for holes in data

In [ ]:
import pandas as pd

path = "/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/weather_station_061055.csv"

df_1 = pd.read_csv(path)

df_1.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load processed station file
path = "/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/weather_station_061078.csv"
df = pd.read_csv(path, parse_dates=["timestamp_utc"])

# Filter to 2004
df_2004 = df[df["timestamp_utc"].dt.year == 2004]

# Extract the station name (same for all rows)
station_name = df["station_name"].iloc[0].strip()


# Create figure
fig, ax1 = plt.subplots(figsize=(14,6))

# Precip left
ax1.plot(
    df_2004["timestamp_utc"],
    df_2004["precip_incremental"],
    color="indigo",
    alpha=0.8,
    label="Precipitation"
)
ax1.set_ylabel("Precipitation", color="indigo")
ax1.tick_params(axis="y", labelcolor="indigo")


# temperature right axis
ax2 = ax1.twinx()
ax2.plot(
    df_2004["timestamp_utc"],
    df_2004["temp"],
    color="royalblue",
    alpha=0.8,
    label="Temperature (C)"
)
ax2.set_ylabel("Temperature (C)", color="royalblue")
ax2.tick_params(axis="y", labelcolor="royalblue")

# Title and layout
plt.title(f"{station_name} - Temperature and Precipitation - 2004 Observations")
plt.tight_layout()
plt.show()


# Trialing with demand and new data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py


In [ ]:
demand.head()

## CSV I made with relative ranks has the lat/lon information included

In [ ]:
import pandas as pd

rank = pd.read_csv(
    "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"
)

rank.head()

In [ ]:
#Extract weather‑station metadata from your processed CSVs

import pandas as pd
import glob

weather_files = glob.glob("/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/*.csv")

weather_meta = []
for f in weather_files:
    df = pd.read_csv(f, nrows=1)
    weather_meta.append({
        "station_id": df["station_id"].iloc[0],
        "station_name": df["station_name"].iloc[0].strip(),
        "lat": df["lat"].iloc[0],
        "lon": df["lon"].iloc[0],
        "file": f
    })

weather = pd.DataFrame(weather_meta)


In [ ]:
#loading substation data

import pandas as pd

subs_unique = rank[["station_code", "station_name", "lat", "lon"]].drop_duplicates()


In [ ]:
#compute nearest weather station for each substation - BallTree

from sklearn.neighbors import BallTree
import numpy as np

# Convert to radians
wx_coords = np.radians(weather[["lat", "lon"]].values)
sub_coords = np.radians(subs_unique[["lat", "lon"]].values)

# Build BallTree with haversine distance
tree = BallTree(wx_coords, metric="haversine")

dist, idx = tree.query(sub_coords, k=1)

subs_unique["nearest_station_id"] = weather.iloc[idx.flatten()]["station_id"].values
subs_unique["nearest_station_file"] = weather.iloc[idx.flatten()]["file"].values
subs_unique["distance_km"] = dist.flatten() * 6371  # convert radians to km


In [ ]:
subs_unique.head()


## Plotting blakehurst relative ranked demand and humidity ANZAC day 2007

In [ ]:
#identify blakehurst nearest weather station
sub_id = "BLAKE"

wx_file = subs_unique.loc[
    subs_unique["station_code"] == sub_id,
    "nearest_station_file"
].iloc[0]


In [ ]:
#loading weather data and filter to ANZAC day 2007

import pandas as pd

wx = pd.read_csv(wx_file, parse_dates=["timestamp_utc"])
wx["date"] = wx["timestamp_utc"].dt.date

anzac = pd.to_datetime("2007-04-25").date()
wx_day = wx[wx["date"] == anzac]


In [ ]:
# loading blakehurst demand for anzac day 2007

rank["date"] = pd.to_datetime(rank["date"]).dt.date

rank_blake = rank[
    (rank["station_code"] == "BLAKE") &
    (rank["date"] == anzac)
].iloc[0]

d00_04 = rank_blake["00_04_mean"]
d04_10 = rank_blake["04_10_mean"]
d10_15 = rank_blake["10_15_mean"]
d15_20 = rank_blake["15_20_mean"]
d20_24 = rank_blake["20_24_mean"]


In [ ]:
#build demand-time sercies aligned to weather timestamps

import numpy as np

demand_series = []

for ts in wx_day["timestamp_utc"]:
    hour = ts.hour
    if 0 <= hour < 4:
        demand_series.append(d00_04)
    elif 4 <= hour < 10:
        demand_series.append(d04_10)
    elif 10 <= hour < 15:
        demand_series.append(d10_15)
    elif 15 <= hour < 20:
        demand_series.append(d15_20)
    else:
        demand_series.append(d20_24)

wx_day["demand_block"] = demand_series


In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12,6))

# Relative humidity (line)
ax1.plot(
    wx_day["timestamp_utc"],
    wx_day["relative_humidity"],
    color="teal",
    label="Relative Humidity (%)"
)
ax1.set_ylabel("Relative Humidity (%)", color="teal")
ax1.tick_params(axis="y", labelcolor="teal")

# Demand blocks (step function)
ax2 = ax1.twinx()
ax2.step(
    wx_day["timestamp_utc"],
    wx_day["demand_block"],
    where="post",
    color="indigo",
    linewidth=2,
    label="Demand (block means)"
)
ax2.set_ylabel("Demand (MW)", color="indigo")
ax2.tick_params(axis="y", labelcolor="indigo")

plt.title("Blakehurst — Demand Blocks and Relative Humidity\nANZAC Day 2007 (00–24h)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Plotting raw demand 2007 ANZAC day blakehurst vs humidity

In [ ]:
#renaming time index for demand to plot

#demand = demand.reset_index()
demand.head()
#demand = demand.rename(columns={"index": "timestamp_utc"})


In [ ]:
#wx["timestamp_utc"] = wx["timestamp_utc"].dt.tz_localize(None)
wx["timestamp_utc"].dtype


In [ ]:
demand["timestamp_utc"] = demand["timestamp_utc"].dt.tz_localize(None)


In [ ]:
demand["timestamp_utc"].dtype

In [ ]:
wx["timestamp_utc"].dtype


In [ ]:
# 1. Rebuild demand_day from the cleaned demand DataFrame
anzac = pd.to_datetime("2007-04-25").date()

demand_day = demand[
    demand["timestamp_utc"].dt.date == anzac
][["timestamp_utc", "BLAKE"]].rename(columns={"BLAKE": "demand_raw"})


In [ ]:
wx_day = wx[
    wx["timestamp_utc"].dt.date == anzac
][["timestamp_utc", "relative_humidity"]]


In [ ]:
merged = demand_day.merge(
    wx_day,
    on="timestamp_utc",
    how="inner"
)


In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12,6))

# Raw demand (left axis)
ax1.plot(
    merged["timestamp_utc"],
    merged["demand_raw"],
    color="indigo",
    linewidth=2,
    label="Raw Demand (MW)"
)
ax1.set_ylabel("Demand (MW)", color="indigo")
ax1.tick_params(axis="y", labelcolor="indigo")

# Relative humidity (right axis)
ax2 = ax1.twinx()
ax2.plot(
    merged["timestamp_utc"],
    merged["relative_humidity"],
    color="teal",
    alpha=0.7,
    linewidth=2,
    label="Relative Humidity (%)"
)
ax2.set_ylabel("Relative Humidity (%)", color="teal")
ax2.tick_params(axis="y", labelcolor="teal")

plt.title("Blakehurst — Raw Demand and Relative Humidity\nANZAC Day 2007 (00–24h)")
plt.xlabel("Time (UTC)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Plotting demand vs precipitation

In [ ]:
wx_day = wx[
    wx["timestamp_utc"].dt.date == anzac
][["timestamp_utc", "precip_incremental"]]


In [ ]:
wx_day = wx_day.rename(columns={"precip_incremental": "precip"})


In [ ]:
merged = demand_day.merge(wx_day, on="timestamp_utc", how="inner")


In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12,6))

# Demand
ax1.plot(
    merged["timestamp_utc"],
    merged["demand_raw"],
    color="indigo",
    linewidth=2
)
ax1.set_ylabel("Demand (MW)", color="indigo")
ax1.tick_params(axis="y", labelcolor="indigo")

# Precipitation
ax2 = ax1.twinx()
ax2.bar(
    merged["timestamp_utc"],
    merged["precip"],
    width=0.02,
    color="cadetblue",
    alpha=0.6
)
ax2.set_ylabel("Precipitation (mm)", color="cadetblue")
ax2.tick_params(axis="y", labelcolor="cadetblue")

plt.title("Blakehurst — Raw Demand and Precipitation\nANZAC Day 2007")
plt.xlabel("Time (UTC)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
